In [1]:
from pathlib import Path
from typing import NamedTuple
from Z0Z_tools import readAudioFile, writeWAV
import numpy

class Deviation(NamedTuple):
	μ: float
	σ: float

good: dict[str, Deviation] = dict(
	primary = Deviation(μ=0.95, σ=0.02),
	close = Deviation(μ=0.10, σ=0.05),
	noise = Deviation(μ=0.01, σ=0.005),
	far = Deviation(μ=0.001, σ=0.0005),
)

bad: dict[str, Deviation] = dict(
	primary = Deviation(μ=0.75, σ=0.20),
	close = Deviation(μ=0.30, σ=0.20),
	noise = Deviation(μ=0.10, σ=0.05),
	far = Deviation(μ=0.03, σ=0.05),
)

pathWrite = Path('/apps/hunterFormsBS/tests/dataSamples/SpeakSoftly_BrokenMan60sec/')

waveformBeta = readAudioFile('/apps/hunterFormsBS/tests/dataSamples/SpeakSoftly_BrokenMan60sec/bass.wav')
waveformCharlie = readAudioFile('/apps/hunterFormsBS/tests/dataSamples/SpeakSoftly_BrokenMan60sec/drums.wav')
waveformEcho = readAudioFile('/apps/hunterFormsBS/tests/dataSamples/SpeakSoftly_BrokenMan60sec/other.wav')
waveformFoxtrot = readAudioFile('/apps/hunterFormsBS/tests/dataSamples/SpeakSoftly_BrokenMan60sec/vocals.wav')


In [ ]:
import random
import scipy.signal

source_waveforms = {
    'bass': waveformBeta,
    'drums': waveformCharlie,
    'other': waveformEcho,
    'vocals': waveformFoxtrot
}

def generate_envelope(length: int, dev: Deviation, theta: float = 0.0001) -> numpy.ndarray:
    """Generate a random walk with regression to the mean (Ornstein-Uhlenbeck process)."""
    # X_{t} = X_{t-1} + theta * (mu - X_{t-1}) + sigma_noise * N(0,1)
    # the steady-state variance is sigma_noise^2 / (2 * theta)
    # so we want sigma_noise = dev.σ * sqrt(2 * theta)

    sigma_noise = dev.σ * numpy.sqrt(2 * theta)
    noise = numpy.random.normal(0, sigma_noise, length)

    # Use scipy.signal.lfilter to generate AR(1) efficiently
    # y[n] = (1 - theta) * y[n-1] + noise[n]
    y = scipy.signal.lfilter([1], [1, -(1 - theta)], noise)

    # Add mean to shift centered process
    envelope = y + dev.μ

    # Prevent negative volumes
    envelope = numpy.clip(envelope, 0.0, None)
    return envelope

def create_mixed_wav(primary_name: str, quality: str, dev_dict: dict[str, Deviation]):
    primary_wav = source_waveforms[primary_name]
    length = primary_wav.shape[-1]

    others = [k for k in source_waveforms.keys() if k != primary_name]
    random.shuffle(others)

    roles = {
        'primary': primary_name,
        'close': others[0],
        'noise': others[1],
        'far': others[2]
    }

    summed_wav = numpy.zeros_like(primary_wav, dtype=numpy.float32)

    for role, name in roles.items():
        dev = dev_dict[role]
        wav = source_waveforms[name]

        envelope = generate_envelope(length, dev)
        summed_wav += wav * envelope

    # Normalize the output to the target primary wav's volume
    target_max = numpy.max(numpy.abs(primary_wav))
    current_max = numpy.max(numpy.abs(summed_wav))

    if current_max > 0:
        summed_wav = summed_wav * (target_max / current_max)

    out_name = f"{primary_name}{quality}.wav"
    out_path = pathWrite / out_name

    writeWAV(out_path, summed_wav)
    print(f"Created {out_name}")

# Randomize independently each time
for primary in ['bass', 'drums', 'other', 'vocals']:
    create_mixed_wav(primary, 'Good', good)
    create_mixed_wav(primary, 'Bad', bad)


Created bassGood.wav
Created bassBad.wav
Created drumsGood.wav
Created drumsBad.wav
Created otherGood.wav
Created otherBad.wav
Created vocalsGood.wav
Created vocalsBad.wav
